In [ ]:
import torch
import torch.nn as nn

class LightGCNEncoder(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super(LightGCNEncoder, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # initial embeddings
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, norm_adj):
        """
        inputs:
            norm_adj = normalized adjacency matrix
        outputs:
            final_user_embeddings
            final_item_embeddings
        """
        # layer 0 embddings
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        
        # concat of user and items embeddings to make complete matrix
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        embs = [all_emb]

        # message passing
        for layer in range(self.num_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)

        # layer combination
        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        # splitting final embeddings into user embeddings and item embeddings
        final_user_embeddings, final_item_embeddings = torch.split(
            final_emb, [self.num_users, self.num_items]
        )

        return final_user_embeddings, final_item_embeddings